In [1]:
# 01_preprocess.py (PATCHED)
# Pré-processamento para modelagem (CNN / BiLSTM)
# - Limpeza e renomeios
# - Dummies para categorias de movimento (se presentes)
# - Taxa de missing por participante
# - Forward/backward fill por participante/estímulo
# - TimeDelta
# - Fixation_Duration opcional (se existir dummy de Fixation)
# - Features derivadas (Δ e velocidades)
# - Padronização e export
# - Salva também a lista de features usadas
# - Converte colunas numéricas para float **antes** de ffill/bfill e de calcular diffs
# - Evita FutureWarning de downcasting e corrige TypeError em diffs com strings

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

df = pd.read_csv("merged_eye_tracking_metadata.csv", low_memory=False)

# 1) Renomeios (se existirem)
if "Category" in df.columns:
    df.rename(columns={"Category": "Eye Movement Category"}, inplace=True)
if "Stimulus" in df.columns:
    df.rename(columns={"Stimulus": "Presented Stimulus name"}, inplace=True)

# 2) Remover registros sem 'Class'
df = df.dropna(subset=["Class"])

# 3) Mapear target
df["Class_numeric"] = df["Class"].map({"TD": 0, "ASD": 1}).astype(int)

# 4) Colunas numéricas base
numeric_cols = [
    "RecordingTime [ms]",
    "Pupil Diameter Right [mm]",
    "Pupil Diameter Left [mm]",
    "Point of Regard Right X [px]",
    "Point of Regard Right Y [px]",
    "Point of Regard Left X [px]",
    "Point of Regard Left Y [px]",
    "Gaze Vector Right X",
    "Gaze Vector Right Y",
    "Gaze Vector Right Z",
    "Gaze Vector Left X",
    "Gaze Vector Left Y",
    "Gaze Vector Left Z",
    "Pupil Size Right X [px]",
    "Pupil Size Right Y [px]",
    "Pupil Size Left X [px]",
    "Pupil Size Left Y [px]",
    "Eye Position Right X [mm]",
    "Eye Position Right Y [mm]",
    "Eye Position Right Z [mm]",
    "Eye Position Left X [mm]",
    "Eye Position Left Y [mm]",
    "Eye Position Left Z [mm]",
    "Pupil Position Right X [px]",
    "Pupil Position Right Y [px]",
    "Pupil Position Left X [px]",
    "Pupil Position Left Y [px]",
]

# 4.1) Coagir **já** as colunas numéricas para float, para evitar object/strings
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 5) Dummies de categorias (se existirem)
categorical_cols = [c for c in ["Category Right", "Category Left"] if c in df.columns]
if len(categorical_cols):
    df = pd.get_dummies(
        df, columns=categorical_cols, prefix=categorical_cols, drop_first=True
    )
dummy_cols = [
    c
    for c in df.columns
    if c.startswith("Category Right_") or c.startswith("Category Left_")
]

# 6) Taxa de missing por participante nas features (exceto tempo)
features_for_missing = [
    c for c in numeric_cols if c != "RecordingTime [ms]"
] + dummy_cols
missing_features = (
    df.groupby("Participant")[features_for_missing]
    .apply(lambda x: x.isnull().mean())
    .reset_index()
)
missing_features = missing_features.rename(
    columns={col: col + "_missing_rate" for col in features_for_missing}
)
df = df.merge(missing_features, on="Participant", how="left")

# 7) Preenchimento temporal por grupo (as colunas já estão em float)
group_cols = ["Participant"]
if "Presented Stimulus name" in df.columns:
    group_cols = ["Participant", "Presented Stimulus name"]

cols_to_fill = [c for c in numeric_cols if c != "RecordingTime [ms]"] + dummy_cols
for c in cols_to_fill:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")  # reforço
df[cols_to_fill] = df.groupby(group_cols)[cols_to_fill].transform(
    lambda x: x.ffill().bfill()
)

# 8) Ordenar e calcular TimeDelta
df = df.sort_values(by=["Participant", "RecordingTime [ms]"]).reset_index(drop=True)
df["TimeDelta [ms]"] = df.groupby("Participant")["RecordingTime [ms]"].diff().fillna(0)

# 9) Fixation_Duration opcional (se existir alguma dummy *_Fixation)
fixation_cols = [c for c in dummy_cols if c.endswith("_Fixation")]
derived_cols = ["TimeDelta [ms]"]
if len(fixation_cols) > 0:
    fx_col = fixation_cols[0]
    df["Event_Group"] = (df[fx_col] != df[fx_col].shift()).cumsum()
    event_durations = (
        df.groupby(["Participant", "Event_Group"])["TimeDelta [ms]"]
        .sum()
        .reset_index(name="Fixation_Duration [ms]")
    )
    df = df.merge(event_durations, on=["Participant", "Event_Group"], how="left")
    derived_cols.append("Fixation_Duration [ms]")


# 10) Features derivadas (Δ e velocidades) por participante — com dtype garantido
def _diff_group(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.diff().fillna(0.0)


if {"Point of Regard Right X [px]", "Point of Regard Right Y [px]"}.issubset(
    df.columns
):
    df["dPOR_Right_X"] = df.groupby("Participant")[
        "Point of Regard Right X [px]"
    ].transform(_diff_group)
    df["dPOR_Right_Y"] = df.groupby("Participant")[
        "Point of Regard Right Y [px]"
    ].transform(_diff_group)
    df["POR_Right_speed"] = np.sqrt(df["dPOR_Right_X"] ** 2 + df["dPOR_Right_Y"] ** 2)
else:
    for c in ["dPOR_Right_X", "dPOR_Right_Y", "POR_Right_speed"]:
        df[c] = 0.0

if {"Point of Regard Left X [px]", "Point of Regard Left Y [px]"}.issubset(df.columns):
    df["dPOR_Left_X"] = df.groupby("Participant")[
        "Point of Regard Left X [px]"
    ].transform(_diff_group)
    df["dPOR_Left_Y"] = df.groupby("Participant")[
        "Point of Regard Left Y [px]"
    ].transform(_diff_group)
    df["POR_Left_speed"] = np.sqrt(df["dPOR_Left_X"] ** 2 + df["dPOR_Left_Y"] ** 2)
else:
    for c in ["dPOR_Left_X", "dPOR_Left_Y", "POR_Left_speed"]:
        df[c] = 0.0

if "Pupil Diameter Right [mm]" in df.columns:
    df["dPupil_Right"] = df.groupby("Participant")[
        "Pupil Diameter Right [mm]"
    ].transform(_diff_group)
else:
    df["dPupil_Right"] = 0.0
if "Pupil Diameter Left [mm]" in df.columns:
    df["dPupil_Left"] = df.groupby("Participant")["Pupil Diameter Left [mm]"].transform(
        _diff_group
    )
else:
    df["dPupil_Left"] = 0.0

delta_speed_cols = [
    "dPOR_Right_X",
    "dPOR_Right_Y",
    "POR_Right_speed",
    "dPOR_Left_X",
    "dPOR_Left_Y",
    "POR_Left_speed",
    "dPupil_Right",
    "dPupil_Left",
]

# 11) Padronização (StandardScaler) nas features numéricas (exceto tempo cru)
features_to_scale = [c for c in numeric_cols if c != "RecordingTime [ms]"]
features_to_scale += [c + "_missing_rate" for c in features_for_missing]
features_to_scale += [c for c in derived_cols if c != "TimeDelta [ms]"]
features_to_scale += delta_speed_cols

# garantir float
for col in features_to_scale + ["TimeDelta [ms]"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
df[features_to_scale] = df[features_to_scale].fillna(0.0)

# 12) EDA rápida (opcional)
try:
    plt.figure(figsize=(6, 4))
    sns.countplot(x="Class_numeric", data=df)
    plt.title("Distribuição da Classe pós pré-processamento")
    plt.xticks([0, 1], ["TD", "ASD"])
    plt.tight_layout()
    plt.savefig("preprocess_distribuicao_classe.png", dpi=150)
    plt.close()
except Exception as e:
    print("Aviso: não foi possível gerar gráfico (headless).", e)

# 13) Exportar dataset e lista de features
out_csv = "merged_preprocessed_for_model.csv"
df.to_csv(out_csv, index=False)

features_final = []
numeric_for_model = [c for c in numeric_cols if c != "RecordingTime [ms]"]
features_final += numeric_for_model
features_final += [c + "_missing_rate" for c in features_for_missing]
features_final += [
    c for c in derived_cols
]  # inclui "TimeDelta [ms]" e possivelmente "Fixation_Duration [ms]"
features_final += delta_speed_cols
features_final += dummy_cols

# Remover duplicatas mantendo ordem
seen = set()
features_final = [x for x in features_final if not (x in seen or seen.add(x))]

with open("features_list.json", "w", encoding="utf-8") as f:
    json.dump(features_final, f, ensure_ascii=False, indent=2)

print(f"✅ Pré-processamento concluído ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print(f"✅ Arquivo salvo: {out_csv}")
print(f"✅ Lista de features salva: features_list.json (total={len(features_final)})")

✅ Pré-processamento concluído (2026-07-14 12:42)
✅ Arquivo salvo: merged_preprocessed_for_model.csv
✅ Lista de features salva: features_list.json (total=82)
